# scRNA-seq: Quality Control and Preprocessing

**Tier 3 — Applied Bioinformatics | Module 30 · Notebook 1**

*Prerequisites: Module 03 (RNA-seq Analysis), Module 07 (Machine Learning for Biology)*

---

**By the end of this notebook you will be able to:**
1. Explain the scRNA-seq experimental workflow (droplet-based 10x Genomics vs plate-based Smart-seq)
2. Generate a count matrix from raw FASTQ files using Cell Ranger or STARsolo
3. Perform quality control: filter cells by UMI count, gene count, and % mitochondrial reads
4. Normalize (library-size, scran) and log-transform count data
5. Identify highly variable genes (HVGs) for downstream analysis


**Key resources:**
- [Scanpy tutorials](https://scanpy.readthedocs.io/en/stable/tutorials.html)
- [Seurat tutorials](https://satijalab.org/seurat/articles/pbmc3k_tutorial)
- [Harvard HBC scRNA-seq training](https://hbctraining.github.io/scRNA-seq_online/)
- [Best practices for scRNA-seq (Luecken & Theis 2019)](https://www.embopress.org/doi/full/10.15252/msb.20188746)

<!-- COPILOT_NOTEBOOK_ONBOARDING_V1 -->
## Why this notebook matters in real projects
This workflow maps to production-style applied bioinformatics: pipeline design, model evaluation, and biological interpretation for decision-making.


## How to start using this notebook
1. Run the **Quick starter data demo** cell below first to confirm your environment and file paths.
2. Then run notebook sections top-to-bottom once, and only then start modifying parameters.
3. Keep one small, known-good example input while experimenting; compare each output against it.
4. For larger or real data, copy the same workflow and replace only input paths first.


## Complicated moments explained
These are common points where learners usually get stuck:
- Pipeline outputs are context-dependent: QC failures upstream can invalidate downstream interpretation.
- Batch effects and confounders can dominate signal if not controlled explicitly.
- Use uncertainty-aware decisions: rank candidates and keep rationale for every threshold.


## Quick starter data demo (run this first)
This cell loads tiny synthetic files from `Course/Assets/data/notebook_starters/` so you can see real input/output behavior immediately.


In [ ]:
# Quick starter demo: load tiny example files and inspect outputs
from pathlib import Path
import json
import csv

root = Path.cwd().resolve()
while root != root.parent and not (root / 'Course').exists():
    root = root.parent

base = root / 'Course' / 'Assets' / 'data' / 'notebook_starters'
print(f'Using starter data folder: {base}')

# CSV preview (DNA)
with open(base / 'dna_sequences.csv', newline='') as f:
    rows = list(csv.DictReader(f))
print('dna_sequences.csv (first 3 rows):')
for row in rows[:3]:
    print(row)

# Variants preview
with open(base / 'variants.csv', newline='') as f:
    variants = list(csv.DictReader(f))
print('\nvariants.csv (first 3 rows):')
for row in variants[:3]:
    print(row)

# FASTA preview
print('\nprotein_sequences.fasta (headers):')
for line in (base / 'protein_sequences.fasta').read_text().splitlines():
    if line.startswith('>'):
        print(line)

# Metadata preview
meta = json.loads((base / 'sample_metadata.json').read_text())
print('\nMetadata keys:', list(meta.keys()))


---

[← Module Overview](../README.md) | [Next: Dimensionality Reduction & Clustering →](./02_dimensionality_reduction.ipynb)

---

## 1. scRNA-seq Technology Overview

> Droplet-based (10x Chromium, Drop-seq) vs plate-based (Smart-seq2, CEL-seq2). UMI concept and deduplication. Library complexity and capture efficiency. Typical cell numbers and read depth recommendations.

## 2. Generating Count Matrices

> Cell Ranger `count` workflow (alignment → barcode detection → UMI counting). STARsolo as open-source alternative. Output: barcodes.tsv, features.tsv, matrix.mtx (Market Exchange format). Show how to load with `scanpy.read_10x_mtx`.

In [ ]:
# Example: Load a 10x count matrix
# import scanpy as sc
# adata = sc.read_10x_mtx('data/pbmc3k/filtered_feature_bc_matrix/', var_names='gene_symbols')
# adata.var_names_make_unique()
# print(adata)

## 3. Quality Control Metrics

> Compute per-cell metrics: `n_genes_by_counts`, `total_counts`, `pct_counts_mt`. Violin plots for each metric. Scatter plot: total_counts vs n_genes colored by pct_counts_mt. Threshold selection strategies (MAD-based, manual inspection).

In [ ]:
# Example: QC filtering
# sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)
# sc.pl.violin(adata, ['n_genes_by_counts', 'total_counts', 'pct_counts_mt'], jitter=0.4, multi_panel=True)
# adata = adata[adata.obs.n_genes_by_counts > 200, :]
# adata = adata[adata.obs.pct_counts_mt < 20, :]

## 4. Normalization and Log-Transformation

> Library-size normalization to 10,000 counts per cell (`sc.pp.normalize_total`). Log1p transformation. scran pooling-based normalization as alternative. Brief mention of SCTransform (Seurat) and analytic Pearson residuals.

In [ ]:
# Example: Normalize
# sc.pp.normalize_total(adata, target_sum=1e4)
# sc.pp.log1p(adata)

## 5. Highly Variable Gene Selection

> Mean-variance relationship in scRNA-seq. `sc.pp.highly_variable_genes` with Seurat flavor vs Cell Ranger flavor. Plot mean expression vs dispersion. How many HVGs to select (2000–5000 typical).

In [ ]:
# Example: Select HVGs
# sc.pp.highly_variable_genes(adata, min_mean=0.0125, max_mean=3, min_disp=0.5)
# sc.pl.highly_variable_genes(adata)

## Summary

> Recap preprocessing pipeline steps. Checklist of decisions made at each step. Save processed AnnData object for next notebook.

In [ ]:
# Example: Save processed object
# adata.write('data/pbmc3k_processed.h5ad')
# print(f'Cells after QC: {adata.n_obs}, Genes: {adata.n_vars}')